# SLAM 02 — ICP: Aligning Two Point Clouds from Scratch

> **Geo-Instructor · SLAM Career Track · Notebook 2 of 3**

---

## What you will build

You have two LiDAR scans of the same room taken from slightly different positions.
You will implement **Iterative Closest Point (ICP)** from scratch to align them.

The alignment gives you the robot's **relative pose** between the two scans —
the core of **scan matching**, which is how LiDAR SLAM builds maps.

### You will implement:
1. Naive ICP (brute-force nearest neighbor)
2. KD-tree ICP (fast nearest neighbor)
3. Point-to-Plane ICP (the variant used in most production SLAM systems)
4. FPFH + RANSAC global registration (get a good initial guess first)

---

## Why ICP matters for SLAM jobs

ICP is the workhorse behind:
- **LOAM** (LiDAR Odometry and Mapping — the dominant LiDAR SLAM method)
- **NDT-based SLAM** (Normal Distributions Transform)
- **Scan-to-map matching** in most AV SLAM pipelines
- **3D reconstruction** (Kinect Fusion, TSDF integration)

In SLAM interviews: "Explain ICP" comes up constantly.

---

```
Runtime: ~3 min  |  No GPU needed  |  numpy + scipy only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from scipy.spatial import KDTree
from scipy.linalg import svd
import time

np.random.seed(7)
plt.rcParams.update({
    'figure.facecolor': '#F4F2F0',
    'axes.facecolor':   '#F4F2F0',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'monospace',
})
print('Ready.')

---
## Part 1 — The Problem: Two Scans, Unknown Relative Pose

A 2-D LiDAR scans the same room twice. The robot moved between scans by:
- translation `(tx, ty)`
- rotation `theta`

We observe two sets of points `P` and `Q`. We want the **rigid transformation** `T`
such that `T(P) ≈ Q`.

```
  [ x' ]   [ cos(t)  -sin(t)  tx ] [ x ]
  [ y' ] = [ sin(t)   cos(t)  ty ] [ y ]
  [ 1  ]   [ 0        0       1  ] [ 1 ]
```

In [ ]:
# ── Generate synthetic 2-D scans of a "room" ─────────────────────────────────

def make_room_scan(n_walls=4, n_pts_per_wall=40, noise=0.02):
    """Simulate a 2-D LiDAR scan of a rectangular room with obstacles."""
    pts = []
    # Outer walls
    for t in np.linspace(0, 1, n_pts_per_wall):
        pts += [[t*6, 0], [t*6, 4], [0, t*4], [6, t*4]]   # rectangle
    # A pillar obstacle
    for t in np.linspace(0, 2*np.pi, 30):
        pts.append([3 + 0.4*np.cos(t), 2 + 0.4*np.sin(t)])
    # A shelf on the wall
    for t in np.linspace(1, 2.5, 20):
        pts.append([t, 3.5])
    pts = np.array(pts, dtype=float)
    pts += np.random.randn(*pts.shape) * noise
    return pts

def apply_transform(pts, theta, tx, ty):
    """Apply a 2-D rigid transform to a point cloud."""
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    return (R @ pts.T).T + np.array([tx, ty])

# Source scan P (reference)
P = make_room_scan(noise=0.03)

# Target scan Q = P transformed by ground-truth pose
TRUE_THETA = 0.15   # ~8.6 degrees
TRUE_TX    = 0.4
TRUE_TY    = -0.2
Q = apply_transform(P, TRUE_THETA, TRUE_TX, TRUE_TY)
Q += np.random.randn(*Q.shape) * 0.03   # add extra noise to Q
# Remove 10% of points from Q (partial overlap, like a real scan)
Q = Q[np.random.choice(len(Q), int(0.9*len(Q)), replace=False)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(P[:,0], P[:,1], s=8, c='steelblue', alpha=0.6, label='P — source scan')
ax.scatter(Q[:,0], Q[:,1], s=8, c='tomato', alpha=0.6, label='Q — target scan (moved)')
ax.set_aspect('equal'); ax.legend()
ax.set_title(f'Two noisy LiDAR scans\nTrue pose: theta={np.degrees(TRUE_THETA):.1f}deg, tx={TRUE_TX}m, ty={TRUE_TY}m')
plt.tight_layout(); plt.show()
print(f'P: {len(P)} points, Q: {len(Q)} points')

---
## Part 2 — ICP Algorithm

```
  Given: source cloud P,  target cloud Q
  Init:  T = identity (or an initial guess)

  repeat:
    1. TRANSFORM: P' = T(P)     (apply current estimate)
    2. CORRESPOND: for each p in P', find nearest neighbor q in Q
    3. MINIMIZE: find the rigid T* that minimizes sum ||p - q||^2
               (closed-form solution via SVD)
    4. UPDATE: T = T* composed with T
    5. CHECK: if change in T is small enough, stop
```

The **SVD step** (step 3) is the mathematical core. Let's derive it:

```
  Given matched pairs {(p_i, q_i)}:
  1. Compute centroids: p_bar = mean(P),  q_bar = mean(Q)
  2. Center the clouds: p_i' = p_i - p_bar,  q_i' = q_i - q_bar
  3. Cross-covariance:  H = sum(p_i' @ q_i'.T)
  4. SVD:               U, S, Vt = svd(H)
  5. Rotation:          R = V @ U.T
  6. Translation:       t = q_bar - R @ p_bar
```

In [ ]:
# ── Core ICP functions ────────────────────────────────────────────────────────

def best_fit_transform(src, dst):
    """
    Given matched point pairs src <-> dst,
    find the optimal rotation R and translation t via SVD.
    Returns: R (2x2), t (2,), rmse
    """
    p_bar = src.mean(axis=0)
    q_bar = dst.mean(axis=0)

    A = src - p_bar
    B = dst - q_bar

    H = A.T @ B           # 2x2 cross-covariance
    U, S, Vt = svd(H)
    R = Vt.T @ U.T

    # Handle reflection (determinant should be +1 for a proper rotation)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = q_bar - R @ p_bar
    residuals = (R @ src.T).T + t - dst
    rmse = np.sqrt(np.mean(np.sum(residuals**2, axis=1)))
    return R, t, rmse


def icp(src, dst, max_iters=50, tol=1e-6, use_kdtree=True, max_dist=None):
    """
    Iterative Closest Point.
    Returns: R_final, t_final, errors (per-iteration RMSE), iters
    """
    src_h = src.copy().astype(float)
    R_total = np.eye(2)
    t_total = np.zeros(2)
    errors  = []
    history = [src_h.copy()]   # for visualization

    kdt = KDTree(dst) if use_kdtree else None

    for it in range(max_iters):
        # 1. Find nearest neighbors
        if use_kdtree:
            dists, indices = kdt.query(src_h)
        else:
            # Brute force O(n*m)
            dists2 = np.sum((src_h[:, None, :] - dst[None, :, :])**2, axis=2)
            indices = np.argmin(dists2, axis=1)
            dists   = np.sqrt(dists2[np.arange(len(src_h)), indices])

        # 2. Optional: discard pairs whose distance exceeds max_dist
        mask = np.ones(len(src_h), dtype=bool)
        if max_dist is not None:
            mask = dists < max_dist
        if mask.sum() < 5:
            break

        matched_src = src_h[mask]
        matched_dst = dst[indices[mask]]

        # 3. Best-fit transform for this iteration
        R, t, rmse = best_fit_transform(matched_src, matched_dst)
        errors.append(rmse)

        # 4. Apply to source
        src_h = (R @ src_h.T).T + t
        history.append(src_h.copy())

        # 5. Accumulate
        R_total = R @ R_total
        t_total = R @ t_total + t

        # 6. Convergence check
        if it > 0 and abs(errors[-1] - errors[-2]) < tol:
            break

    return R_total, t_total, errors, history

print('ICP functions defined.')

In [ ]:
# ── Run ICP: P -> Q alignment ─────────────────────────────────────────────────
t0 = time.time()
R_est, t_est, errors, history = icp(P, Q, max_iters=60, tol=1e-7, max_dist=1.5)
elapsed = time.time() - t0

# Recover estimated angle from rotation matrix
theta_est = np.arctan2(R_est[1, 0], R_est[0, 0])

print(f'ICP converged in {len(errors)} iterations  ({elapsed*1000:.1f} ms)')
print()
print(f'         theta      tx       ty')
print(f'True:    {np.degrees(TRUE_THETA):.3f}deg   {TRUE_TX:.4f}m  {TRUE_TY:.4f}m')
print(f'ICP est: {np.degrees(theta_est):.3f}deg   {t_est[0]:.4f}m  {t_est[1]:.4f}m')
print()
print(f'Final RMSE: {errors[-1]:.5f} m')

In [ ]:
# ── Visualize convergence ─────────────────────────────────────────────────────
P_aligned = history[-1]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ICP Alignment', fontsize=12)

# Before
ax = axes[0]
ax.scatter(P[:,0],       P[:,1],       s=6, c='steelblue', alpha=0.5, label='P source')
ax.scatter(Q[:,0],       Q[:,1],       s=6, c='tomato',    alpha=0.5, label='Q target')
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.set_title('Before: raw scans')

# After
ax = axes[1]
ax.scatter(P_aligned[:,0], P_aligned[:,1], s=6, c='steelblue', alpha=0.5, label='P aligned')
ax.scatter(Q[:,0],          Q[:,1],          s=6, c='tomato',    alpha=0.5, label='Q target')
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.set_title('After ICP: aligned')

# Convergence
ax = axes[2]
ax.semilogy(errors, 'steelblue', lw=2, marker='o', ms=4)
ax.set_xlabel('Iteration'); ax.set_ylabel('RMSE (m, log scale)')
ax.set_title(f'Convergence — {len(errors)} iterations')

plt.tight_layout(); plt.show()

# Mid-run snapshots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
snap_iters = [0, len(history)//4, len(history)//2, len(history)-1]
for ax, it in zip(axes, snap_iters):
    snap = history[it]
    ax.scatter(Q[:,0],    Q[:,1],    s=5, c='tomato',    alpha=0.4, label='Q')
    ax.scatter(snap[:,0], snap[:,1], s=5, c='steelblue', alpha=0.6, label='P')
    ax.set_aspect('equal')
    label = 'Initial' if it == 0 else f'Iter {it}'
    ax.set_title(f'{label}\nRMSE={errors[min(it, len(errors)-1)]:.3f}m', fontsize=8)
    ax.set_xlim(-0.5, 7); ax.set_ylim(-0.5, 5)
    ax.axis('off')
plt.suptitle('ICP snapshots: source cloud converging to target', fontsize=10)
plt.tight_layout(); plt.show()

---
## Part 3 — KD-tree vs Brute Force: Why It Matters at Scale

In real LiDAR SLAM you have 100,000+ points per scan.  
Brute-force nearest neighbor is `O(n*m)` per iteration — completely unusable.
A KD-tree reduces it to `O(n log m)`.

In [ ]:
# ── Benchmark: brute force vs KD-tree ────────────────────────────────────────
sizes = [50, 200, 500, 1000, 3000]
t_brute, t_kd = [], []

for n in sizes:
    rng  = np.random.RandomState(42)
    Ps   = rng.randn(n, 2)
    Qs   = rng.randn(n, 2)

    # Brute force (1 iteration)
    t0 = time.perf_counter()
    d2 = np.sum((Ps[:, None, :] - Qs[None, :, :])**2, axis=2)
    idx = np.argmin(d2, axis=1)
    t_brute.append(time.perf_counter() - t0)

    # KD-tree
    t0 = time.perf_counter()
    kdt = KDTree(Qs)
    _, idx_kd = kdt.query(Ps)
    t_kd.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.loglog(sizes, t_brute, 'o-', color='tomato',    lw=2, label='Brute force O(n^2)')
ax.loglog(sizes, t_kd,    's-', color='steelblue', lw=2, label='KD-tree O(n log n)')
ax.set_xlabel('Number of points (log scale)')
ax.set_ylabel('Time per NN query (s, log scale)')
ax.set_title('Nearest Neighbor Speed: KD-tree vs Brute Force')
ax.legend()
plt.tight_layout(); plt.show()

ratio = t_brute[-1] / t_kd[-1]
print(f'At n={sizes[-1]}: KD-tree is {ratio:.0f}x faster')

---
## Part 4 — Point-to-Plane ICP

The standard ICP minimizes **point-to-point** distance: `||p_i - q_i||^2`.

**Point-to-plane ICP** minimizes the distance along the **surface normal** instead:
```
  minimize  sum  ( (p_i - q_i) . n_i )^2
```
where `n_i` is the surface normal at `q_i`.

This converges **4-10x faster** because it allows sliding along flat surfaces
(walls, floors) without penalty — matching the actual degrees of freedom of
the alignment problem. It is the default in **LOAM**, **LeGO-LOAM**, and **KISS-ICP**.

In [ ]:
# ── Estimate 2-D normals from nearest neighbors ───────────────────────────────
def estimate_normals_2d(pts, k=8):
    """
    Estimate 2-D surface normals via PCA of k-nearest neighbors.
    The normal is the direction of *minimum* variance (eigenvector of smallest eigenvalue).
    """
    kdt = KDTree(pts)
    normals = np.zeros_like(pts)
    for i, p in enumerate(pts):
        _, idx = kdt.query(p, k=k)
        neighbors = pts[idx]
        cov = np.cov(neighbors.T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        # smallest eigenvalue -> normal direction
        normals[i] = eigvecs[:, 0]
    return normals

# Estimate normals on target Q
normals_Q = estimate_normals_2d(Q, k=10)

# Visualize normals on a small subset
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(Q[:,0], Q[:,1], s=6, c='tomato', alpha=0.5)
stride = 5
ax.quiver(Q[::stride,0], Q[::stride,1],
          normals_Q[::stride,0], normals_Q[::stride,1],
          scale=30, width=0.003, color='navy', alpha=0.7, label='normals')
ax.set_aspect('equal')
ax.set_title('Surface normals estimated from local PCA')
plt.tight_layout(); plt.show()
print('Normals shape:', normals_Q.shape)

In [ ]:
# ── Point-to-Plane ICP (linearized Gauss-Newton) ─────────────────────────────

def icp_point_to_plane(src, dst, normals_dst, max_iters=30, tol=1e-7, max_dist=1.5):
    """
    Point-to-plane ICP using linearized least-squares (Gauss-Newton).
    In 2-D: minimize sum [ n_i . (R*p_i + t - q_i) ]^2
    Linearize R ~ I + [0 -a; a 0] for small angle a.
    This gives a 3x3 linear system per iteration: [a, tx, ty] = A^-1 b
    """
    src_h  = src.copy().astype(float)
    kdt    = KDTree(dst)
    errors = []

    for _ in range(max_iters):
        dists, indices = kdt.query(src_h)
        mask = dists < max_dist
        if mask.sum() < 5: break

        ps = src_h[mask]
        qs = dst[indices[mask]]
        ns = normals_dst[indices[mask]]

        # Build linearized system A*x = b  where x = [angle, tx, ty]
        # For each pair: n . (p + [0,-a;a,0]*p + [tx,ty] - q) = 0
        # => n . cross(p) * a + n . [tx,ty] = n . (q - p)
        # cross(p) in 2D = [-py, px] (the skew-symmetric part)
        cross_p = np.column_stack([-ps[:, 1], ps[:, 0]])  # shape (N, 2)
        A_col0  = np.sum(ns * cross_p, axis=1)            # n . cross(p)
        A_col1  = ns[:, 0]                                 # nx
        A_col2  = ns[:, 1]                                 # ny
        A = np.column_stack([A_col0, A_col1, A_col2])      # (N, 3)
        b = np.sum(ns * (qs - ps), axis=1)                 # (N,)

        # Solve least squares
        x, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        angle, tx, ty = x

        # Apply update
        R = np.array([[np.cos(angle), -np.sin(angle)],
                      [np.sin(angle),  np.cos(angle)]])
        src_h = (R @ src_h.T).T + np.array([tx, ty])

        rmse = np.sqrt(np.mean(np.sum((src_h[mask] - qs)**2, axis=1)))
        errors.append(rmse)
        if len(errors) > 1 and abs(errors[-1] - errors[-2]) < tol:
            break

    return src_h, errors


src_p2p, errors_p2p = icp_point_to_plane(P, Q, normals_Q, max_iters=40)
_, _, errors_std, _ = icp(P, Q, max_iters=40)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(errors_std, 'steelblue', lw=2, marker='o', ms=4, label=f'Point-to-Point ({len(errors_std)} iters)')
ax.semilogy(errors_p2p, 'tomato',    lw=2, marker='s', ms=4, label=f'Point-to-Plane  ({len(errors_p2p)} iters)')
ax.set_xlabel('Iteration'); ax.set_ylabel('RMSE (m)')
ax.set_title('Point-to-Plane ICP converges faster than Point-to-Point')
ax.legend()
plt.tight_layout(); plt.show()

print(f'P2Point:  {len(errors_std)} iterations to converge, final RMSE={errors_std[-1]:.5f}m')
print(f'P2Plane:  {len(errors_p2p)} iterations to converge, final RMSE={errors_p2p[-1]:.5f}m')

---
## Part 5 — The Basin of Convergence Problem

ICP is **local** — it gets stuck if the initial misalignment is too large.
This is the core reason SLAM systems need a **global registration** or **loop closure**
step to give ICP a good starting guess.

In [ ]:
# ── Test ICP with increasing initial misalignment ────────────────────────────
init_rotations = np.linspace(0, np.pi, 16)   # 0 to 180 degrees
final_errors   = []

for init_angle in init_rotations:
    # Apply extra rotation to source as initial misalignment
    P_init = apply_transform(P, init_angle, 0, 0)
    _, _, errs, _ = icp(P_init, Q, max_iters=80)
    final_errors.append(errs[-1] if errs else float('nan'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.degrees(init_rotations), final_errors, 'o-', color='steelblue', lw=2)
ax1.axhline(0.1, color='tomato', ls='--', label='"Good alignment" threshold (0.1m)')
ax1.set_xlabel('Initial misalignment (degrees)')
ax1.set_ylabel('Final RMSE (m)')
ax1.set_title('ICP Basin of Convergence')
ax1.legend(fontsize=8)

# Visualize a failure case vs success case
good_init = 0.1   # small rotation
bad_init  = 1.5   # ~86 degrees — outside basin

for ax, angle, label in [(ax2, bad_init, f'Bad init: {np.degrees(bad_init):.0f} deg')]:
    P_test = apply_transform(P, angle, 0, 0)
    _, _, _, hist = icp(P_test, Q, max_iters=60)
    ax.scatter(Q[:,0],         Q[:,1],         s=5, c='tomato',    alpha=0.4, label='Q target')
    ax.scatter(P_test[:,0],    P_test[:,1],     s=5, c='gray',      alpha=0.3, label='P initial')
    ax.scatter(hist[-1][:,0],  hist[-1][:,1],   s=5, c='steelblue', alpha=0.6, label='P after ICP')
    ax.set_aspect('equal'); ax.legend(fontsize=8)
    ax.set_title(f'{label}\n-> ICP diverges (needs global init)')

plt.tight_layout(); plt.show()

basin_end = np.degrees(init_rotations[np.array(final_errors) > 0.15].min()) if any(e > 0.15 for e in final_errors) else 180
print(f'ICP converges reliably up to ~{basin_end:.0f} deg initial misalignment.')
print('=> Real SLAM systems need global registration or IMU pre-integration to seed ICP.')

---
## Part 6 — Exercises

### Exercise 1 — Extend to 3-D
The `best_fit_transform` using SVD generalizes directly to 3-D.  
Change the point generation to 3-D (add z coordinates) and verify the math still works.

### Exercise 2 — Outlier rejection
Add a **trimming step**: after computing correspondences, discard the 20% of pairs
with the largest distance (Trimmed ICP). Does it improve convergence on the
noisy scan?

### Exercise 3 — NDT vs ICP
Normal Distributions Transform (NDT) is an alternative to ICP that represents
the target cloud as a grid of Gaussians instead of individual points.  
Look up the algorithm and implement a 2-D version. Compare convergence speed.

### Exercise 4 — Accumulate poses
Simulate 20 overlapping scan pairs along a path (each scan shifted slightly from the last).
Chain the ICP results together to build a 2-D map. This is the core of **LiDAR odometry**.
Watch drift accumulate over time.

---
## Summary

| Concept | Used in |
|---------|--------|
| ICP predict-update loop | All LiDAR SLAM front-ends |
| SVD closed-form solution | LOAM, KISS-ICP, Open3D |
| KD-tree nearest neighbor | Every production ICP implementation |
| Point-to-plane ICP | LOAM, LeGO-LOAM, HDL-SLAM |
| Basin of convergence | Why factor graphs need loop closure |

**Next:** Notebook 3 — Pose Graph Optimization: Closing the Loop